# Lab 03 — Escalonamento de características e taxa de aprendizado
**Curso 1 · Semana 2 · Regressão linear múltipla — material de estudo em português**

Baseado no laboratório fornecido, *C1_W2_Lab03_Feature_Scaling_and_Learning_Rate_Soln.ipynb*, e nas aulas de Andrew Ng sobre feature scaling, convergência e learning rate.

Este caderno é uma adaptação didática. O notebook do curso depende de arquivos como `lab_utils_multi.py`, `lab_utils_common.py`, `deeplearning.mplstyle` e do conjunto completo de imóveis, não anexados. Por isso, os resultados registrados no original são explicados separadamente; a prática executável usa dados **simulados**, e não reproduz os números exatos do curso. Não são necessárias funções privadas do laboratório.

## O que vamos aprender
- Revisar a regressão linear com várias variáveis e a função de custo.
- Entender por que o gradiente pode atualizar os pesos de modo desigual.
- Comparar taxas de aprendizado pequenas, adequadas e grandes demais.
- Aplicar escalonamento simples, normalização pela média e padronização Z-score.
- Entender como verificar a convergência e prever o preço de um novo imóvel.
- Relacionar as fórmulas matemáticas diretamente com o código NumPy.

Para executar, use Python com NumPy e Matplotlib instalados. Execute as células em ordem.

## 1. Importações

No original, além do NumPy e Matplotlib, há funções importadas de módulos do curso: `load_house_data` carrega os imóveis; `run_gradient_descent` treina o modelo; `plot_cost_i_w` mostra custo e trajetória de um peso; `norm_plot` visualiza distribuições; `plt_equal_scale` compara curvas de nível. Aqui implementamos as operações principais para entender o que acontece por dentro.

- `numpy` permite trabalhar com matrizes, vetores, médias, desvios padrão e produtos escalares.
- `matplotlib.pyplot` é usado para os gráficos.
- `np.set_printoptions(precision=3)` limita a precisão exibida; não altera os cálculos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 2. Problema: prever o preço de uma casa

A regressão linear recebe quatro características (features):

| Área (ft²) | Quartos | Andares | Idade (anos) | Preço (mil dólares) |
|---:|---:|---:|---:|---:|
| 952 | 2 | 1 | 65 | 271,5 |
| 1244 | 3 | 2 | 64 | 232 |
| 1947 | 3 | 2 | 17 | 509,8 |

Estas são as três linhas mostradas no material original, não o conjunto completo. Note que a área agora é em pés quadrados, e não em milhares de pés quadrados, como nos primeiros laboratórios.

### Notação (igual à aula)
- $m$: quantidade de exemplos (linhas).
- $n$: quantidade de características (colunas); aqui $n=4$.
- $\mathbf X$: matriz de treino de dimensão $(m,n)$.
- $\mathbf x^{(i)}$: vetor das características do imóvel $i$.
- $x_j^{(i)}$: característica $j$ do imóvel $i$.
- $y^{(i)}$: preço real do imóvel $i$.
- $\mathbf w=[w_0,w_1,w_2,w_3]$: vetor de pesos, um para cada coluna.
- $b$: intercepto, um único número.
- $f_{\mathbf w,b}$: previsão do modelo.

A célula a seguir cria uma base **simulada** e reproduzível. Os preços foram gerados com uma relação linear e um pouco de ruído; não representam vendas reais. Isso permite experimentar todos os conceitos sem o arquivo do curso.

In [ ]:
rng = np.random.default_rng(42)
m = 100

area = rng.uniform(300, 2000, m)
quartos = rng.integers(1, 6, m)
andares = rng.integers(1, 3, m)
idade = rng.integers(0, 80, m)

# Cada vetor vira uma coluna da matriz X.
X_train = np.column_stack((area, quartos, andares, idade))

# Preço simulado, em milhares de dólares.
y_train = 90 + 0.20 * area + 12 * quartos + 15 * andares - 1.2 * idade
y_train = y_train + rng.normal(0, 20, m)

X_features = ["Área (ft²)", "Quartos", "Andares", "Idade (anos)"]
print("Dimensão de X:", X_train.shape)
print("Dimensão de y:", y_train.shape)
print("Primeira casa:", X_train[0])
print("Preço:", y_train[0])

### O que cada linha está fazendo?
O gerador `rng` usa a semente 42 para produzir sempre a mesma base. `uniform` gera áreas em uma faixa contínua; `integers` gera números inteiros (o limite superior é excluído). `column_stack` reúne os quatro vetores como colunas de uma matriz 100 × 4. O vetor `y_train` contém um preço por casa. O ruído de distribuição normal simula fatores que as quatro características não explicam.

`X_train[0]` retorna a primeira linha. `X_train[:, 0]` retorna a primeira coluna inteira. Os dois pontos significam todas as linhas.

### Visualização de cada característica
Um gráfico de dispersão (scatter plot) desenha um ponto para cada casa. O eixo horizontal mostra uma característica; o vertical, seu preço. No laboratório original, área apresenta associação positiva, idade apresenta associação negativa, e quartos e andares não parecem ter uma relação isolada tão forte. Isso é uma observação visual, não uma prova de causalidade. Em regressão múltipla, uma característica pode ser útil mesmo quando seu gráfico isolado parece pouco informativo.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:, i], y_train, alpha=0.65)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Preço (mil US$)")
plt.tight_layout()
plt.show()

## 3. Regressão linear múltipla: da fórmula ao código

### Modelo com uma variável
$$f_{w,b}(x)=wx+b$$

### Modelo com quatro variáveis
$$f_{\mathbf w,b}(\mathbf x)=w_0x_0+w_1x_1+w_2x_2+w_3x_3+b$$

### Notação vetorial (mesma conta, escrita de forma compacta)
$$f_{\mathbf w,b}(\mathbf x)=\mathbf w\cdot\mathbf x+b$$

O ponto representa o produto escalar: multiplica os elementos de mesma posição e soma os produtos. O vetor $\mathbf w$ tem quatro elementos, mas $b$ continua sendo um escalar.

Exemplo manual com uma casa de 1200 ft², três quartos, um andar e 40 anos, pesos fictícios $[0.2,12,15,-1.2]$ e $b=90$:
$$0.2(1200)+12(3)+15(1)-1.2(40)+90=333$$
A previsão é 333 mil dólares. Estes pesos são apenas ilustrativos; não são os pesos aprendidos.

In [ ]:
x = np.array([1200, 3, 1, 40])
w = np.array([0.2, 12, 15, -1.2])
b = 90

previsao = np.dot(w, x) + b
print("Previsão ilustrativa:", previsao, "mil dólares")

## 4. Função de custo: como medir a qualidade do modelo

$$J(\mathbf w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}(f_{\mathbf w,b}(\mathbf x^{(i)})-y^{(i)})^2$$

Passo a passo:
1. Calcula a previsão de cada imóvel.
2. Subtrai seu valor real: $e^{(i)}=f_{\mathbf w,b}(\mathbf x^{(i)})-y^{(i)}$.
3. Eleva o erro ao quadrado, evitando o cancelamento de erros positivos e negativos.
4. Soma os erros quadráticos de todos os imóveis.
5. Divide por $2m$. O fator 2 facilita a derivação: cancela o 2 que surge ao derivar o quadrado.

A função de custo continua sendo a mesma da regressão com uma variável; o que muda é a fórmula da previsão dentro dos parênteses. Não confunda o erro de um exemplo com o custo do conjunto inteiro. O custo é metade do erro quadrático médio (MSE).

No código, `X @ w` calcula uma previsão por linha e retorna um vetor de dimensão $(m,)$. O escalar `b` é somado a cada elemento. A subtração gera um vetor de erros. `np.mean(erros ** 2)/2` calcula o custo completo.

In [ ]:
def calcular_custo(X, y, w, b):
    previsoes = X @ w + b
    erros = previsoes - y
    custo = np.mean(erros ** 2) / 2
    return custo

print("Custo com parâmetros ilustrativos:", calcular_custo(X_train, y_train, w, b))

## 5. Descida do gradiente com várias variáveis

O objetivo do treinamento é encontrar os pesos e o intercepto que minimizam $J(\mathbf w,b)$.

### Derivadas parciais
$$\frac{\partial J(\mathbf w,b)}{\partial w_j}=\frac{1}{m}\sum_{i=0}^{m-1}(f_{\mathbf w,b}(\mathbf x^{(i)})-y^{(i)})x_j^{(i)}$$

$$\frac{\partial J(\mathbf w,b)}{\partial b}=\frac{1}{m}\sum_{i=0}^{m-1}(f_{\mathbf w,b}(\mathbf x^{(i)})-y^{(i)})$$

A derivada parcial mede a inclinação da função de custo em relação a um parâmetro, mantendo os outros constantes. Não é o tamanho do passo: o passo é a derivada multiplicada pela taxa $\alpha$.

Por que aparece $x_j^{(i)}$ na derivada de $w_j$? Ao derivar $w_jx_j^{(i)}$ em relação a $w_j$, sobra $x_j^{(i)}$. Ao derivar o intercepto $b$, sobra 1, então a segunda expressão não multiplica pela característica.

### Atualização dos parâmetros
$$w_j:=w_j-\alpha\frac{\partial J(\mathbf w,b)}{\partial w_j},\quad j=0,\ldots,n-1$$
$$b:=b-\alpha\frac{\partial J(\mathbf w,b)}{\partial b}$$

Se o gradiente é positivo, o parâmetro diminui. Se é negativo, o parâmetro aumenta. Todos os gradientes devem ser calculados com os mesmos parâmetros antigos antes de atualizar $w$ e $b$.

Exemplo com um único peso: se $w=0.5$, $\alpha=0.1$ e a derivada é $2$, o novo peso é $0.5-0.1(2)=0.3$. Se a derivada fosse $-2$, o novo peso seria $0.7$. A subtração faz o movimento na direção oposta ao gradiente.

In [ ]:
def calcular_gradiente(X, y, w, b):
    m = X.shape[0]
    erros = X @ w + b - y

    # X.T possui formato (n, m).
    # Multiplicar pelos m erros soma erro * característica para cada coluna.
    dj_dw = (X.T @ erros) / m

    # O intercepto recebe apenas a média dos erros.
    dj_db = np.mean(erros)
    return dj_dw, dj_db

w_inicial = np.zeros(X_train.shape[1])
b_inicial = 0.0
dj_dw, dj_db = calcular_gradiente(X_train, y_train, w_inicial, b_inicial)
print("Gradiente dos quatro pesos:", dj_dw)
print("Gradiente de b:", dj_db)

### A função principal do treinamento

`descida_do_gradiente` é a função que reúne o processo completo. Inicializa os pesos em zero, calcula os gradientes, atualiza os parâmetros e guarda o custo depois de cada iteração.

- `np.zeros(X.shape[1])` cria um peso para cada característica.
- O `for` repete o aprendizado pelo número escolhido de iterações.
- `dj_dw, dj_db` são calculados antes de qualquer atualização.
- `historico` guarda o custo; `trajetoria` guarda os pesos para visualização.
- Se houver um valor não finito, interrompemos o treinamento para evitar overflow.
- A tolerância opcional permite encerrar quando a redução não negativa do custo for pequena. Isso é um indício de convergência, não uma garantia isolada.

In [ ]:
def descida_do_gradiente(X, y, iteracoes, alpha, epsilon=None):
    w = np.zeros(X.shape[1])
    b = 0.0
    historico = [calcular_custo(X, y, w, b)]
    trajetoria = [w.copy()]

    for i in range(iteracoes):
        dj_dw, dj_db = calcular_gradiente(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        custo = calcular_custo(X, y, w, b)
        historico.append(custo)
        trajetoria.append(w.copy())

        if not np.isfinite(custo):
            break

        reducao = historico[-2] - historico[-1]
        if epsilon is not None and 0 <= reducao <= epsilon:
            break

    return w, b, np.array(historico), np.array(trajetoria)

## 6. Taxa de aprendizado (learning rate)

$\alpha$ controla o tamanho da atualização. Ela é compartilhada por todos os pesos e pelo intercepto.

- Muito pequena: o custo pode cair, mas lentamente.
- Adequada: o custo diminui de maneira eficiente.
- Muito alta: a atualização pode ultrapassar o ponto mínimo e o custo oscilar ou crescer.

### Resultados registrados no notebook original (dados do curso)
O custo na tabela abaixo é o registrado **depois** de cada atualização, começando na iteração 0.

| $\alpha$ | Custo após primeira atualização | Custo após décima atualização | Interpretação |
|---|---:|---:|---|
| $9.9\times 10^{-7}$ | 95.588,4 | 1.374.350 | Diverge: passos grandes demais |
| $9\times 10^{-7}$ | 66.461,6 | 35.238,5 | Custo cai; peso da área oscila |
| $1\times 10^{-7}$ | 44.231,3 | 2.087,34 | Custo cai e peso da área se aproxima sem oscilar nas 10 iterações |

O primeiro peso ($w_0$) representa a área. No primeiro teste, ele começa aproximadamente em 0,55 e termina em -0,87; a oscilação cresce. No segundo teste, começa em 0,50 e termina em 0,0774; oscila com amplitude decrescente. No terceiro, vai de aproximadamente 0,055 a 0,231. Não confunda a oscilação de um peso com divergência: o custo pode diminuir mesmo quando o peso atravessa seu valor ótimo repetidamente.

### Experimento executável com a base simulada
Como o limite de estabilidade depende dos dados, calculamos uma escala de referência para as taxas da demonstração. A matriz $A$ inclui uma coluna de 1 para representar $b$. Para este problema quadrático, o maior autovalor de $A^TA/m$ é $L$; a descida do gradiente é estável para $0<\alpha<2/L$. Esse cálculo é um complemento didático, não uma exigência da aula. Também testamos as três taxas originais para comparar, mas seus efeitos podem ser diferentes neste conjunto simulado.

In [ ]:
A = np.column_stack((X_train, np.ones(len(X_train))))
L = np.linalg.eigvalsh(A.T @ A / len(A))[-1]

taxas = {
    "Pequena": 0.02 / L,
    "Adequada": 0.9 / L,
    "Alta demais": 2.1 / L
}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for nome, alpha in taxas.items():
    _, _, hist, traj = descida_do_gradiente(X_train, y_train, 30, alpha)
    ax[0].plot(hist, label=f"{nome}: {alpha:.2e}")
    ax[1].plot(traj[:, 0], label=nome)
ax[0].set_yscale("log")
ax[0].set_xlabel("Atualizações")
ax[0].set_ylabel("Custo J (escala log)")
ax[1].set_xlabel("Atualizações")
ax[1].set_ylabel("Peso da área")
for a in ax:
    a.legend()
plt.tight_layout()
plt.show()

for alpha in [9.9e-7, 9e-7, 1e-7]:
    _, _, hist, _ = descida_do_gradiente(X_train, y_train, 10, alpha)
    print(f"alpha={alpha:.2e} | primeiro custo={hist[1]:.2f} | último={hist[-1]:.2f}")

## 7. Por que o tamanho das características atrapalha o treinamento?

Imagine que a área varia de 300 a 2000, enquanto o número de quartos varia de 0 a 5, como nos slides. Ambas participam da mesma fórmula de gradiente, mas seus valores estão em ordens de grandeza diferentes.

$$\frac{\partial J}{\partial w_j}=\frac{1}{m}\sum_i e^{(i)}x_j^{(i)}$$

Para um exemplo com erro 10, área 1200 e 3 quartos, as contribuições são 12.000 para o gradiente da área e 30 para o gradiente dos quartos. Antes de fazer a média, a primeira contribuição é 400 vezes maior. Isso não significa que o resultado final de todas as derivadas sempre terá essa proporção: os erros podem ter sinais diferentes e as contribuições podem se cancelar.

Com uma única taxa $\alpha$, uma atualização segura para o peso da área pode ser pequena demais para os demais pesos. No original, $w_0$ progredia rapidamente, mas os outros pesos demoravam. O escalonamento procura reduzir esse desequilíbrio.

**Atenção:** a normalização altera a representação das entradas, não o preço real nem a informação sobre a casa. Ela também não torna as características automaticamente igualmente importantes para a previsão.

## 8. Três formas de feature scaling

### A. Divisão pelo máximo (características positivas)

$$x'_j=\frac{x_j}{\max(x_j)}$$

Se área=1200 e o máximo=2000, $x'_j=0.6$. Se quartos=3 e máximo=5, $x'_j=0.6$. Os números passam a ter escalas comparáveis.

### B. Min-max
$$x'_j=\frac{x_j-\min(x_j)}{\max(x_j)-\min(x_j)}$$

O mínimo de treino vira 0 e o máximo vira 1. Se a área varia de 300 a 2000:
$$x'=\frac{1200-300}{2000-300}=\frac{900}{1700}\approx0.529$$

Observação: a faixa de min-max nessa fórmula é **[0,1]**, e não [-1,1]. Para [-1,1], usa-se $2x'-1$. Um novo valor fora dos limites de treino pode ficar fora desse intervalo.

### C. Normalização pela média (mean normalization)
$$x'_j=\frac{x_j-\mu_j}{\max(x_j)-\min(x_j)}$$

Nos slides: área de 300 a 2000, média 600. Para 1200:
$$\frac{1200-600}{2000-300}\approx 0.353$$

Área mínima: $(300-600)/1700\approx-0.18$; máxima: $(2000-600)/1700\approx0.82$.

Para quartos, média 2,3 e intervalo de 0 a 5: mínimo $-2.3/5=-0.46$ e máximo $2.7/5=0.54$.

A subtração da média centraliza os dados em zero; dividir pela amplitude aproxima as escalas. Uma coluna constante tem amplitude zero e precisa ser tratada à parte.

In [ ]:
minimos = X_train.min(axis=0)
maximos = X_train.max(axis=0)
medias = X_train.mean(axis=0)
amplitudes = maximos - minimos

# Evita divisão por zero para colunas constantes.
amplitudes_seguras = np.where(amplitudes == 0, 1, amplitudes)
X_minmax = (X_train - minimos) / amplitudes_seguras
X_mean_norm = (X_train - medias) / amplitudes_seguras
print("Primeira linha original:", X_train[0])
print("Min-max:", X_minmax[0])
print("Normalização pela média:", X_mean_norm[0])

## 9. Z-score normalization (padronização)

Esta é a técnica principal do laboratório.

$$x'^{(i)}_j=\frac{x_j^{(i)}-\mu_j}{\sigma_j}$$

### Primeiro: calcular a média de cada coluna
$$\mu_j=\frac{1}{m}\sum_{i=0}^{m-1}x_j^{(i)}$$

### Segundo: calcular a variância e o desvio padrão
$$\sigma_j^2=\frac{1}{m}\sum_{i=0}^{m-1}(x_j^{(i)}-\mu_j)^2$$
$$\sigma_j=\sqrt{\sigma_j^2}$$

A média representa o centro; a variância mede a dispersão quadrática; o desvio padrão é a raiz quadrada da variância, nas mesmas unidades originais da característica.

### Terceiro: subtrair a média e dividir pelo desvio padrão

Exemplo dos slides: área com média $\mu_1=600$ e desvio padrão $\sigma_1=450$.
- Área 300: $(300-600)/450\approx -0.67$.
- Área 600: $(600-600)/450=0$.
- Área 2000: $(2000-600)/450\approx3.11$.

Quartos com média $\mu_2=2.3$ e desvio padrão $\sigma_2=1.4$:
- 0 quartos: $(0-2.3)/1.4\approx-1.64$.
- 5 quartos: $(5-2.3)/1.4\approx1.93$.

A interpretação de um z-score de 2 é que aquele valor está dois desvios padrão acima da média. Um valor -1 está um desvio padrão abaixo. Para colunas não constantes, após a padronização a média é aproximadamente zero e o desvio padrão é 1. O Z-score **não garante** que todos os valores fiquem entre -1 e 1, nem torna necessariamente a distribuição normal.

### Quais escalas são aceitáveis?
A aula sugere buscar valores aproximadamente entre -1 e 1. Faixas entre -3 e 3 ou -0,3 e 0,3 também podem ser adequadas. Valores de -100 a 100, -0,001 a 0,001 ou 98,6 a 105 são exemplos em que reescalonar pode ser útil. Isso é uma heurística, não uma regra rígida.

In [ ]:
def zscore_normalize_features(X):
    # axis=0: calcula uma estatística por coluna (feature).
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)

    # Coluna constante: evita dividir por zero; após centralizar, vira zero.
    sigma_seguro = np.where(sigma == 0, 1, sigma)

    # Broadcasting: subtrai/divide cada coluna pela estatística correspondente.
    X_norm = (X - mu) / sigma_seguro
    return X_norm, mu, sigma_seguro

X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)
print("Médias do treino:", X_mu)
print("Desvios usados:", X_sigma)
print("Médias após normalizar:", X_norm.mean(axis=0))
print("Desvios após normalizar:", X_norm.std(axis=0))
print("Amplitude original:", np.ptp(X_train, axis=0))
print("Amplitude normalizada:", np.ptp(X_norm, axis=0))

### Entendendo a implementação linha por linha
`axis=0` percorre as linhas para obter um valor por coluna; se usássemos `axis=1`, obteríamos uma média por casa, o que não é a padronização desejada.

`mu` e `sigma` têm dimensão (4,), enquanto `X` tem dimensão (100,4). O NumPy aplica as estatísticas às colunas correspondentes automaticamente (broadcasting). `np.std` usa por padrão o divisor m, igual à fórmula dos slides. `np.ptp` é a diferença entre máximo e mínimo, ou amplitude.

A função retorna três coisas, e não apenas a matriz normalizada: precisamos guardar as médias e os desvios para normalizar corretamente imóveis novos. O tratamento de colunas constantes é uma proteção adicional; essas colunas não têm variação útil para estimar um efeito próprio.

### Visualização em três etapas
Vamos reproduzir a ideia visual do laboratório: área versus idade, antes de transformar, depois de subtrair a média, e por último após dividir pelo desvio padrão. Usar a mesma unidade gráfica em x e y (eixos com escala igual) evidencia o desbalanceamento entre características.

In [ ]:
X_centralizado = X_train - X_mu
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
etapas = [(X_train, "Original"), (X_centralizado, "X - média"), (X_norm, "Z-score")]
for i, (dados, titulo) in enumerate(etapas):
    ax[i].scatter(dados[:, 0], dados[:, 3], alpha=0.65)
    ax[i].set_title(titulo)
    ax[i].set_xlabel("Área")
    ax[i].set_ylabel("Idade")
    ax[i].axis("equal")
plt.tight_layout()
plt.show()

A primeira nuvem de pontos fica alongada porque a área varia muito mais do que a idade. Na segunda, ambas estão centradas próximas de zero, mas ainda com escalas diferentes. Na terceira, as escalas são comparáveis. Isso não elimina necessariamente correlações entre as características.

### Distribuições antes e depois
Os histogramas mostram quantos valores pertencem a cada intervalo. Não são gráficos da função de custo. Compare os eixos horizontais de cada coluna antes e depois de normalizar.

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(14, 6))
for i in range(4):
    ax[0, i].hist(X_train[:, i], bins=15)
    ax[0, i].set_title(X_features[i] + " — original")
    ax[1, i].hist(X_norm[:, i], bins=15)
    ax[1, i].set_title(X_features[i] + " — Z-score")
ax[0, 0].set_ylabel("Contagem")
ax[1, 0].set_ylabel("Contagem")
plt.tight_layout()
plt.show()

## 10. Treinar novamente com os dados normalizados

No original, o algoritmo é executado por 1000 iterações com $\alpha=0.1$. Antes da normalização, foram testados valores na ordem de $10^{-7}$. Depois de normalizar, é possível usar uma taxa muito maior nesse problema, pois os gradientes ficam mais equilibrados. Não existe garantia de que 0,1 será a melhor taxa para todo dataset.

Usamos a mesma função de custo e a mesma descida do gradiente. A única mudança nos dados é trocar $X$ por $X_{\text{norm}}$. O vetor de preços $y$ continua na unidade original.

In [ ]:
w_norm, b_norm, hist_norm, traj_norm = descida_do_gradiente(
    X_norm, y_train, iteracoes=1000, alpha=0.1
)

print("Pesos aprendidos:", w_norm)
print("Intercepto:", b_norm)
print("Custo inicial:", hist_norm[0])
print("Custo final:", hist_norm[-1])
print("Gradientes finais:", calcular_gradiente(X_norm, y_train, w_norm, b_norm))

_, _, hist_raw, _ = descida_do_gradiente(X_train, y_train, 1000, 1e-7)
plt.figure(figsize=(9, 4))
plt.plot(hist_raw, label="Sem normalização, alpha=1e-7")
plt.plot(hist_norm, label="Z-score, alpha=0.1")
plt.yscale("log")
plt.xlabel("Atualizações")
plt.ylabel("Custo J (escala log)")
plt.legend()
plt.show()

## 11. Como verificar se a descida do gradiente convergiu

Na aula, o eixo horizontal é o número de iterações e o vertical é $J(\mathbf w,b)$. Não é um gráfico de $w$ nem de $b$ contra o custo. Essa curva acompanha a aprendizagem do modelo ao longo do tempo.

Quando a taxa é suficientemente pequena para esta regressão linear com gradiente de lote completo, espera-se que o custo não aumente após as atualizações. Uma curva que cai e se estabiliza sugere convergência. A quantidade de iterações necessária depende do problema: a aula exemplifica 30, 1000 ou 100.000.

### Critério automático da aula
Escolha $\epsilon=10^{-3}=0.001$.
$$\Delta J=J_{\text{anterior}}-J_{\text{atual}}$$
Se $0\leq\Delta J\leq\epsilon$, podemos considerar a redução pequena e interromper (heurística). Exemplo: custo anterior 12,3456 e atual 12,3450, redução 0,0006. Já se o custo aumentar, não devemos declarar convergência por esse critério.

Uma taxa muito pequena também pode produzir mudanças pequenas prematuramente. Por isso, é útil conferir o histórico, o valor do gradiente e o limite de iterações. Um custo convergido não significa necessariamente que o modelo generaliza bem para casas não vistas.

In [ ]:
w_stop, b_stop, hist_stop, _ = descida_do_gradiente(
    X_norm, y_train, iteracoes=10000, alpha=0.1, epsilon=0.001
)
print("Atualizações realizadas:", len(hist_stop) - 1)
print("Última redução:", hist_stop[-2] - hist_stop[-1])
print("Gradientes no final:", calcular_gradiente(X_norm, y_train, w_stop, b_stop))
plt.plot(hist_stop)
plt.xlabel("Atualizações")
plt.ylabel("Custo J")
plt.title("Curva de convergência")
plt.show()

## 12. Como escolher a taxa de aprendizado

A sequência mostrada na aula é:

$$0.001,\quad0.003,\quad0.01,\quad0.03,\quad0.1,\quad0.3,\quad1$$

Testa-se uma sequência em que cada valor é aproximadamente três vezes o anterior. Compare as curvas com a mesma inicialização e o mesmo número de iterações. Se o custo explode, confira primeiro se a fórmula usa o sinal de menos e se o gradiente está correto; depois reduza $\alpha$. Se cai muito lentamente, experimente aumentar $\alpha$, desde que a redução permaneça estável.

Neste exemplo as taxas são testadas com as entradas normalizadas. Se a meta for selecionar hiperparâmetros para generalização, use validação separada do teste final. Aqui comparamos convergência do custo de treino, não desempenho em dados desconhecidos.

In [ ]:
plt.figure(figsize=(10, 5))
for alpha in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]:
    _, _, hist, _ = descida_do_gradiente(X_norm, y_train, 100, alpha)
    plt.plot(hist, label=f"alpha={alpha}")
plt.yscale("log")
plt.xlabel("Atualizações")
plt.ylabel("Custo J")
plt.legend()
plt.show()

## 13. Previsões versus valores reais

Agora usamos os pesos treinados para calcular as previsões de todas as casas. No original, o gráfico compara preços reais e previstos em relação a cada uma das quatro características.

O cálculo usa `X_norm`, mas o eixo horizontal usa `X_train` (valores originais). Isso deixa os gráficos mais fáceis de interpretar em ft², quartos, andares e anos. Os pontos não precisam formar uma única reta, pois cada previsão depende das quatro entradas ao mesmo tempo.

`X_norm @ w_norm + b_norm` equivale ao laço original que calcula `np.dot(X_norm[i], w_norm) + b_norm` para cada imóvel.

In [ ]:
yp = X_norm @ w_norm + b_norm
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for i in range(4):
    ax[i].scatter(X_train[:, i], y_train, label="Real", alpha=0.6)
    ax[i].scatter(X_train[:, i], yp, label="Previsto", alpha=0.6)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Preço (mil US$)")
ax[0].legend()
plt.tight_layout()
plt.show()

print("RMSE no treino (mil US$):", np.sqrt(np.mean((yp - y_train)**2)))

## 14. Prever o preço de uma casa nova

A casa proposta no curso tem 1200 ft², 3 quartos, 1 andar e 40 anos.

**Não use diretamente as características originais com os pesos treinados em dados normalizados!**

Primeiro transforme as quatro características usando exatamente as médias e os desvios do treino:
$$\mathbf x_{\text{norm}}=\frac{\mathbf x-\boldsymbol\mu}{\boldsymbol\sigma}$$
Depois calcule:
$$\hat y=\mathbf w_{\text{norm}}\cdot\mathbf x_{\text{norm}}+b_{\text{norm}}$$

Não calcule média e desvio padrão apenas da casa nova: ela tem quatro características diferentes, não quatro observações de uma mesma característica. Se tiver conjuntos de validação e teste, ajuste a normalização somente no treino para evitar vazamento de dados.

O resultado deste exemplo é uma previsão do modelo **simulado**. Multiplicamos por 1000 apenas na apresentação, porque os valores de $y$ estão em milhares de dólares.

In [ ]:
x_house = np.array([1200, 3, 1, 40])
x_house_norm = (x_house - X_mu) / X_sigma
preco = np.dot(x_house_norm, w_norm) + b_norm

print("Características originais:", x_house)
print("Características normalizadas:", x_house_norm)
print(f"Preço simulado previsto: US$ {preco * 1000:,.0f}")

## 15. Curvas de nível (contour plot)

Os slides ilustram duas elipses: com as características em escalas muito diferentes, a função de custo pode apresentar curvas de nível alongadas. Cada curva reúne combinações de parâmetros com o mesmo custo. O mínimo fica na parte central.

A descida do gradiente pode atravessar o vale várias vezes, fazendo zigue-zague. Com características em escala semelhante, o contorno geralmente melhora e o algoritmo consegue percorrer o vale mais eficientemente. Z-score não garante contornos perfeitamente circulares: correlações entre características também influenciam a geometria.

A próxima célula desenha um corte bidimensional do custo para os pesos da área e dos quartos, mantendo os outros parâmetros fixos. É um complemento visual, não a função completa de cinco parâmetros. Centralizamos o alvo e fixamos o intercepto apropriado para comparar o formato. Cada eixo está em unidades próprias de pesos antes e depois da transformação.

In [ ]:
# Para isolar a geometria, comparamos as duas primeiras colunas centralizadas.
# Os contornos abaixo são de custo relativo a um ótimo, com outros pesos fixos.
X2_raw = X_centralizado[:, :2]
X2_norm = X_norm[:, :2]

def desenhar_contornos(ax, X2, titulo):
    H = X2.T @ X2 / len(X2)
    # Custo relativo: 1/2 * delta_w.T @ H @ delta_w
    u = np.linspace(-3, 3, 251)
    v = np.linspace(-3, 3, 251)
    U, V = np.meshgrid(u, v)
    Z = 0.5 * (H[0, 0]*U**2 + 2*H[0, 1]*U*V + H[1, 1]*V**2)
    ax.contour(U, V, Z, levels=[0.25, 1, 2, 4, 8, 16])
    ax.set_title(titulo)
    ax.set_xlabel("Deslocamento de w0")
    ax.set_ylabel("Deslocamento de w1")
    ax.set_aspect("equal")

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
desenhar_contornos(ax[0], X2_raw, "Sem escalonamento")
desenhar_contornos(ax[1], X2_norm, "Com Z-score")
plt.tight_layout()
plt.show()

## 16. Resumo para revisão

| Conceito | Para que serve |
|---|---|
| Modelo de regressão | Calcula uma previsão a partir das características e dos parâmetros |
| Função de custo | Mede o erro quadrático agregado do modelo |
| Gradiente | Informa como o custo varia em relação a cada parâmetro |
| Descida do gradiente | Atualiza os parâmetros para reduzir o custo |
| Alpha | Controla o tamanho de cada atualização |
| Feature scaling | Aproxima a escala das características |
| Mean normalization | Centraliza pela média e divide pela amplitude |
| Z-score | Centraliza pela média e divide pelo desvio padrão |
| Convergência | Indício de que o custo está próximo de se estabilizar |
| Predição | Aplica os parâmetros aprendidos a novas entradas pré-processadas |

### Ordem mental para lembrar
Entrada → normalização → previsão → erro → custo → gradientes → atualização de w e b → repetição → previsão de novos dados.

### Exercícios para testar se você realmente entendeu
1. Se a taxa estiver alta demais e a curva subir, o que você deve verificar?
2. Qual é o Z-score de uma área de 1050 com média 600 e desvio 450?
3. Por que a derivada de b não multiplica pelo valor de uma feature?
4. Por que não se deve calcular uma média nova quando se quer prever uma única casa?
5. Troque alpha de 0.1 para 0.001 e compare o custo após 100 iterações.
6. Explique a diferença entre diminuir a função de custo e generalizar para imóveis novos.

### Respostas comentadas
1. Verifique o sinal negativo da atualização, a implementação do gradiente e a taxa; tente uma taxa menor.
2. (1050 - 600)/450 = 1, ou um desvio acima da média.
3. O coeficiente de b na previsão é 1; sua derivada em relação a b é 1.
4. Os pesos aprendidos correspondem à escala do treino; mudar as estatísticas altera o significado das entradas.
5. A taxa pequena tende a exigir mais iterações. Confira no gráfico, pois o comportamento depende dos dados.
6. O custo de treino mede o ajuste aos exemplos usados para aprender; a generalização precisa ser avaliada em exemplos separados.

**Referência:** laboratório do curso de Machine Learning (Andrew Ng, DeepLearning.AI / Stanford Online). O material original informa que os dados de imóveis derivam do Ames Housing dataset, compilado por Dean De Cock. Os dados executáveis deste caderno são simulados e não devem ser confundidos com os dados Ames originais.